# Analytic Stellarator — Multi-section Poincaré & Island Chain Analysis

This notebook demonstrates:
1. Build an analytic helical-ripple stellarator (`pyna.mag.stellarator`)
2. Locate q=4/1 resonant surface
3. Launch field lines; collect crossings on 4 toroidal sections simultaneously
4. Extract island O/X points on each section
5. Multi-panel Poincaré visualisation with island-width bars
6. Summary plot: half_width_R and half_width_psi vs. toroidal angle

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from pyna.mag.stellarator import SimpleStellarartor, simple_stellarator
from pyna.topo.poincare import PoincareMap, ToroidalSection, poincare_from_fieldlines
from pyna.topo.island_extract import extract_island_width
from pyna.flt import FieldLineTracer, get_backend

## 1. Build stellarator model

In [ ]:
# Compact stellarator: R0=3m, r0=0.3m, q goes from 1.1 to 5.0
# q=4/1 resonance near psi~0.74
st = simple_stellarator(
    R0=3.0, r0=0.3, B0=1.0,
    q0=1.1, q1=5.0,
    m_h=4, n_h=4,
    epsilon_h=0.05,   # 5% helical ripple
)
print(st)
R_ax, Z_ax = st.magnetic_axis
print(f'Magnetic axis: R={R_ax:.3f} m  Z={Z_ax:.3f} m')

psi_res_list = st.resonant_psi(4, 1)
print(f'q=4/1 resonant surface at ψ = {[f"{p:.4f}" for p in psi_res_list]}')
psi_res = psi_res_list[0]

In [ ]:
# Plot q-profile
psi_plot = np.linspace(0.02, 0.96, 200)
q_plot = st.q_of_psi(psi_plot)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(np.sqrt(psi_plot), q_plot, 'b-', lw=2)
ax.axhline(4.0, color='r', ls='--', lw=1, label='q = 4/1')
ax.axvline(np.sqrt(psi_res), color='r', ls=':', lw=1)
ax.set_xlabel('S = √ψ'); ax.set_ylabel('q')
ax.set_title('Safety factor profile'); ax.legend()
plt.tight_layout(); plt.show()

## 2. Multi-section Poincaré map

In [ ]:
# Define 4 toroidal sections at phi = k * π/2  (one per field period)
N_sections = 4
phi_sections = np.linspace(0, 2*np.pi, N_sections, endpoint=False)
sections = [ToroidalSection(phi0=float(p)) for p in phi_sections]
print(f'Sections at phi = {np.degrees(phi_sections).round(1)} deg')

# Start points: bracket the q=4/1 surface
start_pts = st.start_points_near_resonance(4, 1, n_lines=16, delta_psi=0.06)
print(f'Launching {len(start_pts)} field lines...')

In [ ]:
%%time
pmap = poincare_from_fieldlines(
    st.field_func, start_pts, sections=sections,
    t_max=600.0, dt=0.04,
)

for i, phi in enumerate(phi_sections):
    n_cross = len(pmap.crossing_array(i))
    print(f'  Section phi={np.degrees(phi):.1f}°: {n_cross} crossings')

## 3. Extract O/X points on each section

In [ ]:
island_chains = []

for i, phi in enumerate(phi_sections):
    pts = pmap.crossing_array(i)
    if len(pts) < 8:
        print(f'Section {i} (phi={np.degrees(phi):.1f}°): too few points ({len(pts)}), skipping')
        island_chains.append(None)
        continue

    # Filter to points near the resonant surface
    r_pts = np.sqrt((pts[:, 0] - R_ax)**2 + pts[:, 1]**2)
    r_res = np.sqrt(psi_res) * st.r0
    mask = np.abs(r_pts - r_res) < 0.5 * st.r0
    pts_near = pts[mask] if mask.sum() >= 8 else pts

    chain = extract_island_width(
        pts_near[:, :2], R_ax, Z_ax,
        mode_m=4,
        psi_func=lambda R, Z: float(st.psi_ax(R, Z)),
    )
    island_chains.append(chain)
    print(f'Section {i} phi={np.degrees(phi):.1f}°: '
          f'O-pts={len(chain.O_points)}  '
          f'w_R={chain.half_width_R*100:.2f} cm  '
          f'w_psi={chain.half_width_psi:.4f}')

## 4. Multi-panel visualisation (2×2)

In [ ]:
fig = plt.figure(figsize=(14, 12))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3)
axes = [fig.add_subplot(gs[i//2, i%2]) for i in range(N_sections)]

# Background: psi_ax contours
R_bg = np.linspace(st.R0 - 1.3*st.r0, st.R0 + 1.3*st.r0, 200)
Z_bg = np.linspace(-1.3*st.r0, 1.3*st.r0, 200)
Rg, Zg = np.meshgrid(R_bg, Z_bg)
psi_bg = st.psi_ax(Rg, Zg)

for i, (ax, phi) in enumerate(zip(axes, phi_sections)):
    # Background flux surfaces
    ax.contour(Rg, Zg, psi_bg, levels=np.linspace(0, 1.2, 15),
               colors='lightgray', linewidths=0.5)
    ax.contour(Rg, Zg, psi_bg, levels=[1.0], colors='k', linewidths=1.5)
    ax.contour(Rg, Zg, psi_bg, levels=[psi_res], colors='navy',
               linewidths=0.8, linestyles='--')

    # Poincaré scatter
    pts = pmap.crossing_array(i)
    if len(pts) > 0:
        ax.scatter(pts[:, 0], pts[:, 1], s=1.0, c='steelblue',
                   alpha=0.6, rasterized=True)

    # O/X points and island-width bars
    chain = island_chains[i]
    if chain is not None and len(chain.O_points) > 0:
        ax.scatter(chain.O_points[:, 0], chain.O_points[:, 1],
                   s=40, c='red', marker='o', zorder=5, label='O-pt')
        ax.scatter(chain.X_points[:, 0], chain.X_points[:, 1],
                   s=40, c='blue', marker='x', zorder=5, lw=1.5, label='X-pt')

        # Radial double-headed arrow at each O-point
        for O_pt in chain.O_points:
            dr = O_pt[0] - R_ax
            dz = O_pt[1] - Z_ax
            dist = np.sqrt(dr**2 + dz**2) + 1e-30
            ur, uz = dr/dist, dz/dist
            w = chain.half_width_R
            ax.annotate('',
                xy=(O_pt[0] + w*ur, O_pt[1] + w*uz),
                xytext=(O_pt[0] - w*ur, O_pt[1] - w*uz),
                arrowprops=dict(arrowstyle='<->', color='red', lw=1.5),
            )

    ax.plot(R_ax, Z_ax, '+k', ms=8, mew=1.5)
    ax.set_aspect('equal')
    ax.set_xlabel('R (m)'); ax.set_ylabel('Z (m)')
    ax.set_title(f'φ = {np.degrees(phi):.0f}°', fontsize=11)
    ax.set_xlim(st.R0 - 1.2*st.r0, st.R0 + 1.2*st.r0)
    ax.set_ylim(-1.2*st.r0, 1.2*st.r0)
    if i == 0:
        ax.legend(fontsize=7, loc='upper right')

fig.suptitle(
    f'Analytic stellarator — q=4/1 island chain (ψ_res={psi_res:.3f})\n'
    f'ε_h={st.epsilon_h}, m_h/n_h={st.m_h}/{st.n_h}',
    fontsize=12,
)
plt.savefig('stellarator_multiSection_q41.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: stellarator_multiSection_q41.png')

## 5. Summary plot: island width vs. toroidal angle

In [ ]:
w_R   = [c.half_width_R   if c is not None else np.nan for c in island_chains]
w_psi = [c.half_width_psi if c is not None else np.nan for c in island_chains]
n_O   = [len(c.O_points)  if c is not None else 0       for c in island_chains]

phis_deg = np.degrees(phi_sections)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))

ax1.bar(phis_deg, np.array(w_R)*100, width=60, color='steelblue', alpha=0.8)
ax1.set_xlabel('φ (deg)'); ax1.set_ylabel('Half-width (cm)')
ax1.set_title('Island half-width w_R vs section')
ax1.set_xticks(phis_deg)

ax2.bar(phis_deg, w_psi, width=60, color='tomato', alpha=0.8)
ax2.set_xlabel('φ (deg)'); ax2.set_ylabel('Half-width (Δψ)')
ax2.set_title('Island half-width Δψ vs section')
ax2.set_xticks(phis_deg)

plt.tight_layout(); plt.show()
print('Number of detected O-points per section:', n_O)